# 05. Distribution Analysis: How Are Values Spread Out?
### Primary Analytical Purpose: *How are the values of one variable spread out?*
**Lecture Reference:** Purpose 5 — Distribution (Slides 25–28, 45)

Distribution analysis investigates the center, spread, shape, skewness, and outliers of a continuous quantitative variable. It addresses questions such as:
- What is the typical value, and is the data symmetrical or skewed?
- Are observations tightly clustered or broadly dispersed?
- Are there extreme outliers or multiple distinct peaks (multimodality)?

---

## 1. Scenario and Dataset Preview
**Business Scenario:** A quick-commerce delivery platform in Qatar analyzes 450 customer grocery orders. The dataset tracks `order_value_qar` (in QAR), `delivery_time_min` (in minutes), and `service_type` (*Standard* vs. *Express* delivery).

The dataset is loaded from `data/qatar_ecommerce_orders.csv`. Let us inspect the summary statistics.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

# Set modern Seaborn theme
sns.set_theme(style="whitegrid")

# Load orders dataset
df = pd.read_csv("data/qatar_ecommerce_orders.csv")

# Display summary statistics
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
display(df.head())
print("
--- Summary Statistics ---")
display(df[["order_value_qar", "delivery_time_min"]].describe())

## 2. Primary Visualization: Histogram with Median (The Default)
- **What the chart shows:** How many orders fall into each value range. Height represents frequency (count of orders), not monetary amount.
- **When it is useful:** Default choice for understanding the distribution shape of a single continuous variable (Slide 25).
- **Critical Rule from Lecture (Slide 26 & 27):**
  - **With right-skewed data, report the median and IQR, not just the mean!** The mean is pulled up by the heavy tail.
  - Bin width is an analytical choice: try several widths to balance signal vs. noise.

In [ ]:
# Calculate central tendency metrics
median_val = df["order_value_qar"].median()
mean_val = df["order_value_qar"].mean()

# --- Seaborn Implementation: Histogram with Central Tendency Reference Lines ---
plt.figure(figsize=(9, 4.8))

ax = sns.histplot(
    data=df, 
    x="order_value_qar", 
    bins=25, 
    color="#14A098"
)

# Add vertical dashed lines for median and mean
ax.axvline(median_val, color="#D64545", linestyle="--", linewidth=2.5, 
           label=f"Median: {median_val:.1f} QAR")
ax.axvline(mean_val, color="#333333", linestyle=":", linewidth=2, 
           label=f"Mean: {mean_val:.1f} QAR")

# Principle 2: Title states the finding; axis labels carry units
ax.set_title("Order Values Are Right-Skewed: Half of All Orders Are Under 65 QAR", fontsize=13, weight="bold")
ax.set_xlabel("Order Value (QAR)", fontsize=11)
ax.set_ylabel("Number of Orders", fontsize=11)
ax.legend(frameon=True, fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# --- Plotly Express Implementation: Histogram with Marginal Box Plot ---
# Lecture Slide 28: marginal='box' stacks a box plot directly above the histogram!
fig = px.histogram(
    df, 
    x="order_value_qar", 
    nbins=25, 
    marginal="box",
    title="Order Values Are Right-Skewed: Half of All Orders Are Under 65 QAR",
    labels={"order_value_qar": "Order Value (QAR)"}
)

fig.update_traces(marker_color="#14A098")
fig.update_layout(yaxis_title="Number of Orders")
fig.show()

### Structured Interpretation: The 5-Step Framework
1. **Question:** What is the typical grocery basket value for Qatar quick-commerce, and is there a high-value customer tail?
2. **Visualization:** Histogram with 25 bins, marked with median and mean reference lines, plus marginal box plot.
3. **Observation:** Order values are heavily concentrated between 35 and 80 QAR. The median is ~64.8 QAR while the mean is ~74.5 QAR. A thin tail extends past 140 QAR up to 210 QAR.
4. **Insight:** The distribution exhibits marked right-skew. The arithmetic average is inflated by high-value outliers; quoting the mean would mislead management about typical customer spend.
5. **Implication:** Set the free delivery threshold at 80 QAR (just above the median) to incentivize basket expansion without discouraging typical shoppers.

## 3. Alternative 1: Box Plot (Comparing Groups Compactly)
- **What the chart shows:** Five-number summary (Min, Q1, Median, Q3, Max) and individual outlier points.
- **When to use instead:** Best for comparing distributions across multiple categories or groups compactly (Slide 25 & 45).

In [ ]:
# --- Seaborn Implementation: Grouped Box Plot ---
plt.figure(figsize=(8, 4.5))

ax_box = sns.boxplot(
    data=df, 
    x="service_type", 
    y="delivery_time_min", 
    palette=["#14A098", "#E69F00"],
    width=0.45
)

ax_box.set_ylim(0, 80)
ax_box.set_title("Express Delivery Achieves Tight 23-Min Median vs. 46-Min Standard", fontsize=12, weight="bold")
ax_box.set_xlabel("Service Type", fontsize=11)
ax_box.set_ylabel("Delivery Time (Minutes)", fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# --- Plotly Express: Box Plot ---
fig_box = px.box(
    df, 
    x="service_type", 
    y="delivery_time_min", 
    color="service_type",
    color_discrete_map={"Standard": "#14A098", "Express": "#E69F00"},
    points="outliers",
    title="Express Delivery Achieves Tight 23-Min Median vs. 46-Min Standard",
    labels={"service_type": "Service Type", "delivery_time_min": "Delivery Time (Minutes)"}
)
fig_box.show()

### Interpretation for Box Plot
- **Observation:** Express delivery has a median of 23 minutes with a narrow Interquartile Range (IQR ≈ 6 minutes). Standard delivery has a median of 46 minutes with a much wider IQR (≈ 12 minutes) and several outliers exceeding 65 minutes.
- **Derived Insight:** Express operations maintain high predictability and operational consistency, whereas Standard delivery experiences high variance during peak traffic windows.

## 4. Alternative 2: Violin Plot with Strip Plot (Shape & Points)
- **What the chart shows:** Kernel density estimate (showing the full continuous probability shape) combined with jittered individual data points.
- **When to use instead:** When distribution shape (such as bimodal peaks) matters or when sample size allows displaying raw observations (Slide 25 & 45).
- **Why box plots fall short here:** A box plot hides multimodality; a bimodal variable looks completely normal in a box plot (Slide 27).

In [ ]:
# --- Seaborn Implementation: Violin + Strip Plot Overlay ---
plt.figure(figsize=(8.5, 4.8))

# Outer violin shows smooth density silhouette
sns.violinplot(
    data=df, 
    x="service_type", 
    y="delivery_time_min", 
    inner=None, 
    color="#EBF2FA"
)

# Strip plot shows raw observations with jitter and opacity
sns.stripplot(
    data=df, 
    x="service_type", 
    y="delivery_time_min", 
    size=4, 
    alpha=0.4, 
    palette=["#14A098", "#E69F00"]
)

plt.ylim(0, 80)
plt.title("Violin & Strip Plot: Full Density Curves and Raw Observation Points", fontsize=12, weight="bold")
plt.xlabel("Service Type", fontsize=11)
plt.ylabel("Delivery Time (Minutes)", fontsize=11)

plt.tight_layout()
plt.show()

### Interpretation for Violin & Strip Plot
- **Observation:** The Express violin has a single sharp, unimodal bulge centered tightly at 22–24 minutes. The Standard violin exhibits a broader, flatter profile with dots dispersed across 25 to 70 minutes.
- **Derived Insight:** Showing raw points reassures the analyst that the Express service SLA (under 30 minutes) is consistently met for over 95% of orders.

## 5. Best Practices & Common Mistakes (Lecture Principles)
**Visualization Best Practices (Slide 27 & 46):**
- **Try several bin widths:** Too few bins hides structure; too many shows random noise.
- **Label axes clearly:** Ensure frequency count is never mistaken for currency amount.
- **Report median and IQR alongside skewed distributions** (Principle 5 & Slide 48).
- **Use a box plot when comparing many groups;** use a histogram for detailed inspection of one.
- **State the sample size:** Always mention how many observations the distribution represents ($n=450$).

**Common Mistakes & Misleading Designs:**
- **Bars with gaps in a histogram:** Makes a continuous histogram look like a categorical bar chart.
- **Reporting only the mean:** When data is skewed, the arithmetic mean misrepresents the typical experience.
- **Using a box plot on bimodal data:** A box plot compresses two peaks into a misleading middle box.

## 6. Hands-on Practice Exercises

### Exercise 1: Chart Interpretation
Calculate the 25th percentile ($Q_1$), 50th percentile (Median), and 75th percentile ($Q_3$) for `order_value_qar` using `df['order_value_qar'].quantile([0.25, 0.50, 0.75])`.
1. **Observation:** What is the IQR ($Q_3 - Q_1$)?
2. **Insight:** What percentage of orders fall within this middle 50% range?
3. **Implication:** How can marketers target the top 10% orders ($> 90\text{th}$ percentile)?

### Exercise 2: Code Modification
Re-plot the histogram of `order_value_qar` using 10 bins instead of 25. Observe how the distribution profile changes.

In [ ]:
# Exercise 2: Plot histogram with 10 bins
# plt.figure(figsize=(8, 4))
# sns.histplot(data=df, x='order_value_qar', bins=10, color='#14A098')
# ...

### Exercise 3: Filtered Distribution
Filter `df` to `service_type == 'Express'`. Plot a Seaborn histogram of Express delivery times with a vertical line marking the 30-minute target SLA.

In [ ]:
# Exercise 3: Write your Express delivery time distribution code below:
# express_df = df[df['service_type'] == 'Express']
# ...

## 7. Key Takeaways
1. **Histograms reveal center, spread, skew, and modal peaks** for a single continuous measurement.
2. **Median beats mean for skewed data:** Never rely on the arithmetic mean alone when a heavy tail is present.
3. **Box plots excel at compact group comparisons**, while violin plots reveal multimodality and detailed density shapes.
4. **State the sample size ($n$) and units** on every distribution graphic.